In [ ]:
# Imports and data loading

import pandas as pd
import matplotlib.pyplot as plt

import os
import shutil
import random
import cv2

import kagglehub

path = kagglehub.dataset_download(
    "fareselmenshawii/license-plate-dataset"
)

print(path)

# Exploration

import os

for root, dirs, files in os.walk(path):
    print(root)
    print(files[:5])

## see the photos and xmls

missing = []

for img_name in file_names:

    xml_name = img_name.replace(".png", ".xml")

    xml_path = os.path.join(
        annot_folder_path,
        xml_name
    )

    if not os.path.exists(xml_path):
        missing.append(img_name)

print(missing)

# train test split

import os

folders = [
    f"{base_dir}images/train",
    f"{base_dir}images/val",
    f"{base_dir}labels/train",
    f"{base_dir}labels/val"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

train_img_folder = f"{base_dir}images/train"
val_img_folder = f"{base_dir}images/val"

train_label_folder = f"{base_dir}annots/train"
val_label_folder = f"{base_dir}annots/val"

from sklearn.model_selection import train_test_split

train_imgs, temp_imgs = train_test_split(
    file_names,
    test_size=0.2,
    random_state=42
)

val_imgs, test_imgs = train_test_split(
    temp_imgs,
    test_size=0.5,
    random_state=42
)

def copy_images(image_list, destination_folder):

    for img_name in image_list:

        source = os.path.join(
            img_folder_path,
            img_name
        )

        destination = os.path.join(
            destination_folder,
            img_name
        )

        shutil.copy(source, destination)

print(os.path.join(img_folder_path, img_name))
print(os.path.join(annot_folder_path, xml_name))

for img_name in train_imgs:

    xml_name = img_name.replace(".png", ".xml")

    shutil.copy(
        os.path.join(img_folder_path, img_name),
        os.path.join(train_img_folder, img_name)
    )

    shutil.copy(
        os.path.join(annot_folder_path, xml_name),
        os.path.join(train_label_folder, xml_name)
    )

for img_name in val_imgs:

    xml_name = img_name.replace(".png", ".xml")

    shutil.copy(
        os.path.join(img_folder_path, img_name),
        os.path.join(val_img_folder, img_name)
    )

    shutil.copy(
        os.path.join(annot_folder_path, xml_name),
        os.path.join(val_label_folder, xml_name)
    )

print(len(os.listdir(train_img_folder)))
print(len(os.listdir(train_label_folder)))

for img_name in test_imgs:

    xml_name = img_name.replace(".png", ".xml")

    shutil.copy(
        os.path.join(img_folder_path, img_name),
        os.path.join(test_img_folder, img_name)
    )

    shutil.copy(
        os.path.join(annot_folder_path, xml_name),
        os.path.join(test_label_folder, xml_name)
    )

print(len(os.listdir(train_img_folder)))
print(len(os.listdir(train_label_folder)))
print(len(os.listdir(test_img_folder)))
print(len(os.listdir(test_label_folder)))
print(len(os.listdir(val_img_folder)))
print(len(os.listdir(val_label_folder)))

imgs = set(
    f.replace(".png", "")
    for f in os.listdir(train_img_folder)
)

xmls = set(
    f.replace(".xml", "")
    for f in os.listdir(train_label_folder)
)

print(imgs == xmls)

# Convert for YOLO

def convert_bbox(
    xmin,
    ymin,
    xmax,
    ymax,
    img_width,
    img_height
):

    x_center = (xmin + xmax) / 2 / img_width
    y_center = (ymin + ymax) / 2 / img_height

    width = (xmax - xmin) / img_width
    height = (ymax - ymin) / img_height

    return (
        x_center,
        y_center,
        width,
        height
    )

import xml.etree.ElementTree as ET

def xml_to_yolo(xml_path, txt_path):

    tree = ET.parse(xml_path)
    root = tree.getroot()

    img_width = int(root.find("size/width").text)
    img_height = int(root.find("size/height").text)

    lines = []

    for obj in root.findall("object"):

        bbox = obj.find("bndbox")

        xmin = int(bbox.find("xmin").text)
        ymin = int(bbox.find("ymin").text)
        xmax = int(bbox.find("xmax").text)
        ymax = int(bbox.find("ymax").text)

        x_center, y_center, width, height = convert_bbox(
            xmin,
            ymin,
            xmax,
            ymax,
            img_width,
            img_height
        )

        lines.append(
            f"0 {x_center} {y_center} {width} {height}"
        )

    with open(txt_path, "w") as file:
        file.write("\n".join(lines))

xml_to_yolo(
    "/kaggle/input/car-plate-detection/annotations/Cars87.xml",
    "Cars87.txt"
)

with open("Cars87.txt") as f:
    print(f.read())

base_dir = "/kaggle/working/car_plate_dataset"

folders = [
    f"{base_dir}/labels/train",
    f"{base_dir}/labels/val",
    f"{base_dir}/labels/test"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

yolo_train_folder = f"{base_dir}/labels/train"

for xml_name in os.listdir(train_label_folder):

    xml_path = os.path.join(
        train_label_folder,
        xml_name
    )

    txt_name = xml_name.replace(".xml", ".txt")

    txt_path = os.path.join(
        yolo_train_folder,
        txt_name
    )

    xml_to_yolo(xml_path, txt_path)

yolo_test_folder = f"{base_dir}/labels/test"

for xml_name in os.listdir(test_label_folder):

    xml_path = os.path.join(
        test_label_folder,
        xml_name
    )

    txt_name = xml_name.replace(".xml", ".txt")

    txt_path = os.path.join(
        yolo_test_folder,
        txt_name
    )

    xml_to_yolo(xml_path, txt_path)

yolo_val_folder = f"{base_dir}/labels/val"

for xml_name in os.listdir(val_label_folder):

    xml_path = os.path.join(
        val_label_folder,
        xml_name
    )

    txt_name = xml_name.replace(".xml", ".txt")

    txt_path = os.path.join(
        yolo_val_folder,
        txt_name
    )

    xml_to_yolo(xml_path, txt_path)

print(len(os.listdir(yolo_train_folder)))
print(len(os.listdir(yolo_test_folder)))
print(len(os.listdir(yolo_val_folder)))

# yaml file for YOLO

import yaml

data = {
    'path': '/kaggle/input/license-plate-dataset',

    'train': 'images/train',
    'val': 'images/val',

    'names':{
      '0': 'licence_plate'
    }
}

# Open a file and write the data
with open('dataset.yaml', 'w') as file:
    yaml.dump(data, file, default_flow_style=False, sort_keys=False)

print("YAML file written successfully")


with open("dataset.yaml") as f:
    print(f.read())

import os

print(os.path.exists("/kaggle/working/license-plate-dataset/images/train"))
print(os.path.exists("/kaggle/working/license-plate-dataset/labels/train"))

# YOLO

!pip install -U ultralytics

from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.train(
    data="dataset.yaml",
    epochs=50,
    imgsz=640,
    batch=16
)

results = model.predict(
    source=test_path,
    save=True
)

results = model.predict(
    source=test_path,
    save=True
)

import cv2

def extract_plate(image_path):

    img = cv2.imread(image_path)
    img = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2RGB
    )

    results = model(image_path)

    for r in results:
        for box in r.boxes:

            xmin, ymin, xmax, ymax = (
                box.xyxy[0]
                .cpu()
                .numpy()
                .astype(int)
            )

            plate = img[
                ymin:ymax,
                xmin:xmax
            ]

            return plate

plate = extract_plate(test_path)

plt.figure(figsize=(8,3))
plt.imshow(plate)
plt.axis("off")
plt.show()


import cv2
import matplotlib.pyplot as plt

img = cv2.imread(test_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

results = model(test_path)

for r in results:
    for box in r.boxes:

        xmin, ymin, xmax, ymax = (
            box.xyxy[0]
            .cpu()
            .numpy()
            .astype(int)
        )

        plate = img[ymin:ymax, xmin:xmax]

        plt.figure(figsize=(8,3))
        plt.imshow(plate)
        plt.axis("off")
        plt.show()

!pip install easyocr

import easyocr

def preprocess_plate(plate):

    gray = cv2.cvtColor(
        plate,
        cv2.COLOR_RGB2GRAY
    )

    gray = cv2.resize(
        gray,
        None,
        fx=4,
        fy=4,
        interpolation=cv2.INTER_CUBIC
    )

    gray = cv2.GaussianBlur(
        gray,
        (3, 3),
        0
    )

    _, gray = cv2.threshold(
        gray,
        0,
        255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    return gray

plate = extract_plate(test_path)

gray = preprocess_plate(plate)

plt.imshow(gray, cmap="gray")
plt.axis("off")

reader = easyocr.Reader(['en'])

def ocr_plates(image):

    result = reader.readtext(image)

    for bbox, text, prob in result:
      text = text.upper()
        print(
            f"Текст: {text} "
            f"(Точність: {prob:.2f})"
        )

    return result

ocr_plates(gray)